In [3]:
import torch
torch.cuda.is_available()

False

In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()  # 이거 꼭 있어야됨

token = os.getenv("HF_TOKEN")
login(token)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
import torch
from src.model import dinosplus_classfier
import torch.nn as nn
from transformers import AutoModel
from src.re_make_Gate import inject_gating, GatedOutputProjection
import mlflow

mlflow.set_tracking_uri("sqlite:///mlruns.db")



# ── run_id로 .pth 다운로드 ──
run_id = "65d36d5dbadd4fd2ba8d6b2d55663c2d"
client = mlflow.tracking.MlflowClient()
local_path = client.download_artifacts(run_id, "best dino v3 splus gated last ONLY.pth", ".")

# ── 구조 만들고 가중치 로드 ──
backbone = AutoModel.from_pretrained("facebook/dinov3-vits16plus-pretrain-lvd1689m")
backbone, hooks = inject_gating(backbone, gate_type="elementwise", keep_cls_ungated=True, target_layers=None)
model = dinosplus_classfier(backbone, num=200)

model.load_state_dict(torch.load(local_path))
model.eval()
print("로드 완료!")

/home/hyuksu/deep-learning-study/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:184: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 1: invalid argument (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/235 [00:00<?, ?it/s]

[inject_gating] 총 12개 중 12개 레이어에 G1 gate 적용
  → 적용 인덱스: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  (gate_type=elementwise)


RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
print(model.backbone.layer[0].attention)

In [ ]:
from matplotlib import pyplot as plt
plt.imshow(pil_img)

In [ ]:
from src.attention_deep import run
import torch
# 허깅페이스 데이터셋
from datasets import load_dataset
ds = load_dataset("zh-plus/tiny-imagenet")
pil_img = ds['valid'][0]['image']

run(model, pil_img=pil_img, save_dir="data")